In [ ]:
#@title Upload BoolNet rules file
from google.colab import files
import os, textwrap, csv

# ─────────────────────────────────────────────────────────────
# 1)  Upload / reuse file
# ─────────────────────────────────────────────────────────────
if 'file_rules' in globals() and os.path.exists(file_rules):
    print(f"  Using already loaded file: {file_rules}")
else:
    print(textwrap.dedent("""
        Select your rules file (e.g. 'boolnet_rules_XXX.txt').
        You only need to do this ONCE; the path will be saved in the 'file_rules' variable.
    """))
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError(" You must upload a rules file.")
    file_rules = list(uploaded.keys())[0]
    print(f"  File uploaded and saved in 'file_rules' variable: {file_rules}")

# ─────────────────────────────────────────────────────────────
# 2)  Read rules and calculate N and <K>
# ─────────────────────────────────────────────────────────────
genes   = []        # list of targets
exprs   = []        # list of expressions (factors)

with open(file_rules, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        tgt = row['targets'].strip()
        fac = row['factors'].strip()
        if tgt:                       # discard empty lines
            genes.append(tgt)
            exprs.append(fac)

N = len(genes)

# → translate each expression to extract tokens
tr_table = str.maketrans({'(': ' ', ')': ' ', '&': ' ', '|': ' ', '!': ' '})
k_list = []
for expr in exprs:
    # remove operators and split into tokens
    tokens = expr.translate(tr_table).split()
    # regulators = genes that appear in tokens
    regs   = {tok for tok in tokens if tok in genes}
    k_list.append(len(regs))

K_avg = sum(k_list) / N if N else 0.0

print(f"\n  The network contains {N} nodes.")
print(f"  Average K (mean number of regulators per node): {K_avg:.2f}")

In [ ]:
#@title Package Installation
!apt-get -y install graphviz libgraphviz-dev
!pip install pygraphviz
!pip -q install pydot
!pip -q install dot2tex
!pip install pyvis

In [ ]:
#@title Graph network (SVG + PNG) with Graphviz (anti-overlap)

import re
import os
import networkx as nx
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────
# 0) Output paths with original file prefix
# ─────────────────────────────────────────────────────────────
base_name = os.path.splitext(os.path.basename(file_rules))[0]
out_dir   = os.path.dirname(file_rules) or "."
out_svg   = os.path.join(out_dir, f"{base_name}_gene_regulatory_network.svg")
out_png   = os.path.join(out_dir, f"{base_name}_gene_regulatory_network.png")

# ─────────────────────────────────────────────────────────────
# 1) Read the file and build the directed graph
# ─────────────────────────────────────────────────────────────
G = nx.DiGraph()

#   Regex:   (!)?  optional negation sign attached to the name
token_pat = re.compile(r'(!)?\b([A-Za-z0-9_]+)\b')

with open(file_rules, 'r', encoding='utf-8') as fh:
    for line in fh:
        line = line.strip()
        if not line or line.lower().startswith('targets'):
            continue                    # skip header or empty lines

        target, expr = line.split(',', 1)   # only the first comma
        target = target.strip()
        expr   = expr.strip()

        # extract all factors with their sign
        for neg, factor in token_pat.findall(expr):
            # discard logical words if they appear
            if factor.lower() in {'and', 'or', 'not'}:
                continue
            relation = 'negative' if neg else 'positive'
            color    = 'red' if neg else 'green'
            G.add_edge(factor, target, relation=relation, color=color)

# ─────────────────────────────────────────────────────────────
# 2) Node and edge colors
# ─────────────────────────────────────────────────────────────
special_nodes = {
    "DNA_Damage"     : "red",
    "Drug_Resistance": "pink",
    "Proliferation"  : "pink",
    "Senescence"     : "pink",
    "Apoptosis"      : "pink"
}
default_node_color = "lightblue"
node_colors = [special_nodes.get(n, default_node_color) for n in G.nodes()]
edge_colors = [G[u][v]['color'] for u, v in G.edges()]

# ─────────────────────────────────────────────────────────────
# 3) Graphviz layout (anti-overlap)
# ─────────────────────────────────────────────────────────────

try:
    from networkx.drawing.nx_agraph import graphviz_layout as agraph_layout
    _HAVE_AGRAPH = True
except Exception:
    _HAVE_AGRAPH = False

try:
    from networkx.drawing.nx_pydot import graphviz_layout as pydot_layout
    _HAVE_PYDOT = True
except Exception:
    _HAVE_PYDOT = False

if not (_HAVE_AGRAPH or _HAVE_PYDOT):
    raise ImportError(
    )

def gv_layout(G, prog="neato", sep=0.25, overlap="false", splines="true"):
    args = []
    if overlap is not None:
        args.append(f"-Goverlap={overlap}")
    if sep is not None:
        args.append(f"-Gsep=+{sep}")
    if splines is not None:
        args.append(f"-Gsplines={splines}")
    args_str = " ".join(args)

    if _HAVE_AGRAPH:
        return agraph_layout(G, prog=prog, args=args_str)
    else:
        return pydot_layout(G, prog=prog, args=args_str)

# Choose the layout program (try "sfdp" if your network is large)
PROG = "neato"     # options: "neato", "sfdp", "fdp", "dot"
pos = gv_layout(G, prog=PROG, sep=0.25, overlap="false", splines="true")

# ─────────────────────────────────────────────────────────────
# 4) Draw and save the graph (SVG + PNG)
# ─────────────────────────────────────────────────────────────
plt.figure(figsize=(20, 20))
nx.draw(
    G, pos,
    with_labels=True,
    node_color=node_colors,
    edge_color=edge_colors,
    node_size=1700,
    font_size=10,
    font_weight='bold',
    arrowsize=20
)
plt.margins(0.05)

# Save both formats before displaying
plt.savefig(out_svg, format="svg", bbox_inches="tight")
plt.savefig(out_png, format="png", dpi=300, bbox_inches="tight")
plt.show()

print(f"SVG saved to: {out_svg}")
print(f"PNG saved to: {out_png}")


# To graph the attraction basin, use only the 9-node network

In [ ]:
# @title Basins & Attractors global & per-basin maps (PNG/SVG + HTML)

import csv, os, sys, re, glob, zipfile, math, json, shutil
from collections import defaultdict
import networkx as nx
import matplotlib.pyplot as plt
from networkx.drawing.nx_pydot import graphviz_layout

# ── Output root ─────────────────────────────────────────
OUTPUT_DIR = "Basin_Attraction"  # << all artifacts land here

# ── General config ─────────────────────────────────────
GENERATE_GLOBAL_MAP = True        # global map (PNG/SVG + HTML)
GENERATE_BASIN_MAPS = False        # per-basin maps (PNG/SVG + HTML)
CLEAN_LEGACY        = True        # remove previous outputs for same prefix
FIGSIZE             = (12, 12)

# Colors (no metric gradients). Distinct colors per attractor using priority palette.
PALETTE = ['red', 'green', 'blue', 'cyan', 'magenta', 'yellow', 'black', 'orange', 'purple']
EDGE_COLOR_HTML     = "#000000"    # edge color in HTML
CYCLE_BORDER_COLOR  = "#000000"    # border for nodes on the cycle
NONCYCLE_BORDER     = "#222222"

# Labels on PNG/SVG (e.g., state bits). Default False.
SHOW_NUM_LABELS     = False
LABEL_MODE          = "bits"       # "bits" | "index"

# ── Interactivity (HTML) ───────────────────────────────
ALLOW_NODE_DRAG      = "all"       # "none" | "transients" | "all"
HTML_HEIGHT          = "900px"
HTML_WIDTH           = "100%"

# ── Cycle-centric layout for per-basin maps ────────────
CYCLE_CENTRIC_LAYOUT = True        # center the cycle on per-basin maps
CYCLE_LAYOUT_MODE    = "tree"      # "tree" (hierarchical) or "simple"
LAYOUT_R0            = 2.0          # cycle radius
LAYOUT_LAYER_GAP     = 0.7          # distance between transient layers
LAYOUT_WEDGE_FACTOR  = 0.85         # angular amplitude per branch (0..1)

# ── Input: rules file (targets,factors) ────────────────
try:
    file_rules  # if you set this before running, it is respected
except NameError:
    file_rules = "Osteo_mod1_red.csv"  # change to your file

base_rules = os.path.splitext(os.path.basename(file_rules))[0]

# Track everything we generate so we can zip it at the end
created_files = []

# ───────────────────────────────────────────────────────
# 1) Load rules (targets,factors) and prepare genes
# ───────────────────────────────────────────────────────
rules_raw = {}
with open(file_rules, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        tgt = (row.get('targets') or '').strip()
        fac = (row.get('factors') or '').strip()
        if tgt:
            rules_raw[tgt] = fac

if not rules_raw:
    raise RuntimeError("No rules found. Check headers 'targets,factors' and separators.")

genes  = list(rules_raw.keys())
n      = len(genes)
idx_of = {g:i for i,g in enumerate(genes)}

# ───────────────────────────────────────────────────────
# 2) Normalize & compile (FAST + FALLBACK)
# ───────────────────────────────────────────────────────

def normalize_expr(expr: str) -> str:
    if expr is None: return 'False'
    s = str(expr)
    s = (s.replace('\u00a0',' ').replace('\t',' ')
           .replace('\r',' ').replace('\n',' ')
           .replace('\ufeff','').replace('\u200b','')
           .replace('\u200c','').replace('\u200d',''))
    s = re.sub(r'\s+', ' ', s).strip()
    return s if s else 'False'

_token_pat = re.compile(r'\b[A-Za-z_][A-Za-z0-9_]*\b')

def _translate_ops(expr: str) -> str:
    return expr.replace('&',' and ').replace('|',' or ').replace('!',' not ')

def _vars_in_expr(expr: str):
    toks = set(_token_pat.findall(expr))
    return [g for g in genes if g in toks]

def compile_expr_fast(expr: str):
    s = _translate_ops(normalize_expr(expr))
    for g in sorted(genes, key=len, reverse=True):
        s = re.sub(rf'\b{re.escape(g)}\b', f'st[{idx_of[g]}]', s)
    s = f'({s})'
    return compile(s, '<expr>', 'eval')


def compile_expr_fallback(expr: str):
    s = _translate_ops(normalize_expr(expr))
    used = _vars_in_expr(s)
    s = f'({s})'
    return compile(s, '<expr>', 'eval'), used

# Prepare evaluators (one per gene/target)
evaluators = []  # ('fast', code) or ('slow', (code, used_genes))
fallback_count = 0
for g in genes:
    expr = rules_raw[g]
    try:
        code = compile_expr_fast(expr)
        evaluators.append(('fast', code))
    except Exception:
        code, used = compile_expr_fallback(expr)
        evaluators.append(('slow', (code, used)))
        fallback_count += 1


def next_state(st):
    loc_fast = {'st': st}
    out = []
    for kind, payload in evaluators:
        if kind == 'fast':
            code = payload
            val = eval(code, {}, loc_fast)
        else:
            code, used = payload
            loc_slow = {g: bool(st[idx_of[g]]) for g in used}
            val = eval(code, {}, loc_slow)
        out.append(1 if bool(val) else 0)
    return tuple(out)

# ───────────────────────────────────────────────────────
# 3) States, transition & graph (2^n)
# ───────────────────────────────────────────────────────

def int_to_tuple(x, n):
    return tuple((x >> i) & 1 for i in range(n-1, -1, -1))

state_list = [int_to_tuple(i, n) for i in range(1 << n)]
state2next = {s: next_state(s) for s in state_list}

G = nx.DiGraph()
G.add_edges_from((s, state2next[s]) for s in state_list)

# Predecessors (useful for layouts)
preds = defaultdict(list)
for s in state_list:
    preds[state2next[s]].append(s)

# ───────────────────────────────────────────────────────
# 4) Attractors/basins and reordering
# ───────────────────────────────────────────────────────
state2attr     = {}
attractor2info = {}
attr_id_seq    = 0

for init in state_list:
    if init in state2attr: continue
    visited, trajectory = {}, []
    s = init
    while s not in visited and s not in state2attr:
        visited[s] = len(trajectory)
        trajectory.append(s)
        s = state2next[s]
    if s not in state2attr:
        cycle_start = visited[s]
        cycle       = trajectory[cycle_start:]
        aid         = attr_id_seq; attr_id_seq += 1
        attractor2info[aid] = {"cycle": cycle, "basin": set(cycle)}
        for st in cycle: state2attr[st] = aid
    else:
        aid = state2attr[s]
    for st in trajectory:
        state2attr[st] = aid
        attractor2info[aid]["basin"].add(st)


def sort_key(item):
    aid, info = item
    is_cycle   = (len(info["cycle"]) != 1)
    basin_size = len(info["basin"])
    return (is_cycle, -basin_size)

sorted_items = sorted(attractor2info.items(), key=sort_key)
# Map old attractor ids to sequential indices starting at 1
id_map = {old_aid: new_idx for new_idx, (old_aid, _) in enumerate(sorted_items, start=1)}


def basin_id_of_state(s):
    return id_map[state2attr[s]]

# Assign distinct colors per (reordered) attractor using priority palette
ATTRACTOR_COLORS = {new_idx: PALETTE[(new_idx-1) % len(PALETTE)] for new_idx in range(1, len(sorted_items)+1)}

# ───────────────────────────────────────────────────────
# 5) Optional cleanup
# ───────────────────────────────────────────────────────
if CLEAN_LEGACY:
    try:
        shutil.rmtree(OUTPUT_DIR)
    except Exception:
        pass
    for p in glob.glob(f"{base_rules}_Basin_Attraction.zip"):
        try: os.remove(p)
        except: pass

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ───────────────────────────────────────────────────────
# 6) Basin CSVs and summary (no metric columns)
# ───────────────────────────────────────────────────────
basins_dir   = os.path.join(OUTPUT_DIR, "Basins")
separate_dir = os.path.join(OUTPUT_DIR, "basins_sep")
os.makedirs(basins_dir, exist_ok=True)

header       = ["Gene"] + [f"Attractor_{i}" for i in range(1, len(sorted_items)+1)]
summary_rows = [[] for _ in genes]

print(f"\n Found {len(sorted_items)} unique attractors (reordered):\n")
for new_idx, (aid, info) in enumerate(sorted_items, start=1):
    cycle      = info["cycle"]
    basin      = info["basin"]
    basin_size = len(basin)
    kind       = "fixed point" if len(cycle) == 1 else f"limit cycle of length {len(cycle)}"

    basin_csv = os.path.join(basins_dir, f"{base_rules}_attractor_{new_idx:02d}_basin.csv")
    with open(basin_csv, "w", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(["basin"] + genes)
        for state in sorted(basin):
            row = [new_idx] + list(state)
            wr.writerow(row)
    created_files.append(basin_csv)

    print(f"  Attractor {new_idx} ({kind}): {basin_size} configurations → {basin_csv}")

    for g_idx, _ in enumerate(genes):
        if new_idx == 1: summary_rows[g_idx] = [genes[g_idx]]
        bits = ''.join(str(state[g_idx]) for state in cycle)
        summary_rows[g_idx].append(bits)

summary_file_pref = os.path.join(OUTPUT_DIR, f"{base_rules}_attractors_summary_sync.csv")
with open(summary_file_pref, "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(header); wr.writerows(summary_rows)
created_files.append(summary_file_pref)

col_widths = [max(len(str(x)) for x in col) for col in zip(header, *summary_rows)]
fmt_row    = ' '.join(f'{{:<{w}}}' for w in col_widths)
print("\n  Attractors table (sorted by type and basin size):\n")
print(fmt_row.format(*header))
for row in summary_rows: print(fmt_row.format(*row))
print(f"\n  File saved: {summary_file_pref}")

# ───────────────────────────────────────────────────────
# 7) Visualizations (no metrics)
# ───────────────────────────────────────────────────────

def _layout_positions(Gref):
    try:
        pos = graphviz_layout(Gref, prog="twopi")
    except Exception:
        print("  Could not use 'graphviz_layout'; falling back to spring_layout.", file=sys.stderr)
        pos = nx.spring_layout(Gref, seed=42)
    return pos


def _setup_axes(ax, pos):
    xs = [pos[v][0] for v in pos]
    ys = [pos[v][1] for v in pos]
    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)
    padx = 0.03 * (xmax - xmin if xmax > xmin else 1.0)
    pady = 0.03 * (ymax - ymin if ymax > ymin else 1.0)
    ax.set_xlim(xmin - padx, xmax + padx)
    ax.set_ylim(ymin - pady, ymax + pady)
    ax.set_aspect('equal'); ax.axis('off')


def _scale_pos(pos, scale=1200.0):
    xs = [xy[0] for xy in pos.values()]
    ys = [xy[1] for xy in pos.values()]
    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)
    dx = max(xmax - xmin, 1e-9)
    dy = max(ymax - ymin, 1e-9)
    s  = scale / max(dx, dy)
    cx, cy = (xmin + xmax) / 2.0, (ymin + ymax) / 2.0
    return {k: ((v[0] - cx) * s, -(v[1] - cy) * s) for k, v in pos.items()}


def _make_hover_text(node, in_cycle):
    bits = ''.join(str(b) for b in node)
    conf = ', '.join(f"{g}={node[idx_of[g]]}" for g in genes)
    basin_id = basin_id_of_state(node)
    cyc  = " [in cycle]" if in_cycle else ""
    return f"Attractor: {basin_id}{cyc} | State: {bits}, Config: {conf}"

# ======== Cycle-centered layouts ========

def _layout_cycle_centric(G_sub, cycle_order):
    k = len(cycle_order)
    if k == 0:
        return _layout_positions(G_sub)
    two_pi = 2.0 * math.pi
    thetas = [two_pi * i / k for i in range(k)]
    theta_map = {node: th for node, th in zip(cycle_order, thetas)}
    pos = {node: (LAYOUT_R0*math.cos(th), LAYOUT_R0*math.sin(th)) for node, th in theta_map.items()}

    root, level = {}, {}
    for node in G_sub.nodes():
        if node in theta_map:
            root[node], level[node] = node, 0
        else:
            steps, x = 0, node
            while x not in theta_map:
                x = state2next[x]; steps += 1
            root[node], level[node] = x, steps

    from collections import defaultdict as _dd
    groups = _dd(list)
    for node in G_sub.nodes():
        if node not in theta_map:
            groups[(root[node], level[node])].append(node)

    sector = (two_pi / k) if k > 1 else two_pi
    wedge  = sector * LAYOUT_WEDGE_FACTOR

    for (r, d), nodes in groups.items():
        th_r = theta_map[r]
        m = len(nodes)
        offsets = [0.0] if m == 1 else [ (i-(m-1)/2.0)*(wedge/max(m-1,1)) for i in range(m) ]
        radius = LAYOUT_R0 + LAYOUT_LAYER_GAP*d
        for off, node in zip(offsets, nodes):
            th = th_r + off
            pos[node] = (radius*math.cos(th), radius*math.sin(th))
    return pos


def _layout_cycle_centric_tree(G_sub, cycle_order):
    from collections import defaultdict as _dd
    k = len(cycle_order)
    if k == 0:
        return _layout_positions(G_sub)
    two_pi = 2.0 * math.pi
    thetas = [two_pi * i / k for i in range(k)]
    theta_map = {node: th for node, th in zip(cycle_order, thetas)}

    root, level = {}, {}
    for node in G_sub.nodes():
        if node in theta_map:
            root[node], level[node] = node, 0
        else:
            steps, x = 0, node
            while x not in theta_map:
                x = state2next[x]; steps += 1
            root[node], level[node] = x, steps

    children = _dd(list)
    for u in G_sub.nodes():
        v = state2next[u]
        if v in G_sub and not (u in theta_map and v in theta_map):
            children[v].append(u)

    subtree_size = {}
    def dfs_size(x):
        if x in subtree_size: return subtree_size[x]
        total = 1
        for c in children.get(x, []): total += dfs_size(c)
        subtree_size[x] = total
        return total
    for r in cycle_order:
        for c in children.get(r, []): dfs_size(c)

    sector = (two_pi / k) if k > 1 else two_pi
    wedge_total = sector * LAYOUT_WEDGE_FACTOR

    pos = {node: (LAYOUT_R0*math.cos(th), LAYOUT_R0*math.sin(th)) for node, th in theta_map.items()}

    def place_branch(parent, left, right):
        ch = children.get(parent, [])
        if not ch: return
        ch = sorted(ch, key=lambda z: (-subtree_size.get(z,1), z))
        total = sum(subtree_size.get(z,1) for z in ch)
        cur = left
        for z in ch:
            span = (right-left) * (subtree_size.get(z,1)/max(total,1))
            a, b = cur, cur+span
            th = (a+b)/2.0
            r  = LAYOUT_R0 + LAYOUT_LAYER_GAP*level[z]
            pos[z] = (r*math.cos(th), r*math.sin(th))
            place_branch(z, a, b)
            cur = b

    for r in cycle_order:
        center = theta_map[r]
        left   = center - wedge_total/2.0
        right  = center + wedge_total/2.0
        place_branch(r, left, right)
    return pos
# ======== END cycle-centered layouts ========


def write_interactive_html(
    G_sub,
    filename,
    title,
    cycle_nodes=None,
    cycle_order=None,
):
    try:
        from pyvis.network import Network
    except ImportError:
        print(" Missing PyVis. Install with: pip install pyvis")
        return

    if CYCLE_CENTRIC_LAYOUT and cycle_order:
        pos = _layout_cycle_centric_tree(G_sub, list(cycle_order)) if CYCLE_LAYOUT_MODE=="tree" \
              else _layout_cycle_centric(G_sub, list(cycle_order))
    else:
        pos = _layout_positions(G_sub)

    pos_px = _scale_pos(pos, scale=1200.0)

    node_list = list(G_sub.nodes())

    net = Network(height=HTML_HEIGHT, width=HTML_WIDTH, directed=True, bgcolor="#ffffff")
    options = {
        "interaction": {"hover": True, "navigationButtons": True, "dragNodes": True, "dragView": True},
        "physics": {"enabled": False},
        "nodes": {"font": {"size": 0}},
        "edges": {
            "smooth": {"enabled": False},
            "arrows": {"to": {"enabled": True, "scaleFactor": 0.85}},
            "width": 0.8,
            "selectionWidth": 1,
            "hoverWidth": 0,
            "arrowStrikethrough": False,
            "color": {"color": EDGE_COLOR_HTML, "highlight": EDGE_COLOR_HTML, "inherit": False}
        }
    }
    net.set_options(json.dumps(options))

    for node in node_list:
        in_cycle = (cycle_nodes is not None) and (node in cycle_nodes)
        if ALLOW_NODE_DRAG == "all":
            fixed = False
        elif ALLOW_NODE_DRAG == "transients":
            fixed = in_cycle
        else:
            fixed = True

        bg_color = ATTRACTOR_COLORS[basin_id_of_state(node)]
        x, y = pos_px[node]
        net.add_node(
            str(node),
            label=" ",
            title=_make_hover_text(node, in_cycle),
            x=x, y=y, physics=False,
            fixed={"x": True, "y": True} if fixed else False,
            shape="dot", size=8,
            color={"background": bg_color, "border": (CYCLE_BORDER_COLOR if in_cycle else NONCYCLE_BORDER), "borderWidth": (3 if in_cycle else 1)}
        )

    for u, v in G_sub.edges():
        net.add_edge(str(u), str(v))

    net.write_html(filename)
    created_files.append(filename)
    print(f"  Interactive map saved: {filename}")


# ── 7.1) Global map ────────────────────────────────────
if GENERATE_GLOBAL_MAP:
    pos = _layout_positions(G)

    fig = plt.figure(figsize=FIGSIZE); ax  = plt.gca()
    node_colors = [ATTRACTOR_COLORS[basin_id_of_state(node)] for node in G.nodes()]
    nx.draw_networkx_nodes(
        G, pos,
        node_color=node_colors,
        edgecolors='k', linewidths=0.2,
        node_size=120, ax=ax
    )
    nx.draw_networkx_edges(G, pos, arrowstyle='-|>', arrowsize=10, width=1.2, alpha=0.7, node_size=65, ax=ax)

    if SHOW_NUM_LABELS:
        if LABEL_MODE == "bits":
            labels = {node: ''.join(str(b) for b in node) for node in G.nodes()}
        else:
            labels = {node: str(i) for i, node in enumerate(G.nodes())}
        text_objs = nx.draw_networkx_labels(G, pos, labels=labels, font_size=6, ax=ax)
        try:
            import matplotlib.patheffects as pe
            for t in text_objs.values():
                t.set_path_effects([pe.withStroke(linewidth=1.2, foreground="white")])
        except Exception:
            pass

    plt.title('Global basins map (no metrics)')
    _setup_axes(ax, pos); plt.tight_layout()

    def _slugify(s):
        return re.sub(r'[^A-Za-z0-9_\-]+', '_', s)

    slug = _slugify("plain")
    global_svg_pref = os.path.join(OUTPUT_DIR, f"{base_rules}_basins_map_sync_{slug}.svg")
    global_png_pref = os.path.join(OUTPUT_DIR, f"{base_rules}_basins_map_sync_{slug}.png")
    plt.savefig(global_svg_pref, format="svg")
    plt.savefig(global_png_pref, dpi=300)
    plt.close(fig)
    created_files.extend([global_svg_pref, global_png_pref])

    # Global HTML (no metrics)
    all_cycle_nodes = set()
    for info in attractor2info.values():
        all_cycle_nodes.update(info["cycle"])
    interactive_global_html = os.path.join(OUTPUT_DIR, f"{base_rules}_basins_map_sync_interactive_{slug}.html")
    write_interactive_html(
        G,
        interactive_global_html,
        title="Global map (no metrics)",
        cycle_nodes=all_cycle_nodes,
        cycle_order=None
    )

# ── 7.2) Per-basin maps ───────────────────────────────
if GENERATE_BASIN_MAPS:
    os.makedirs(separate_dir, exist_ok=True)

    def draw_basin(aid, info, fname_prefix):
        basin      = info["basin"]
        cycle_list = info["cycle"]        # order within the cycle
        cycle_set  = set(cycle_list)

        H = nx.DiGraph()
        for s in basin:
            H.add_edge(s, state2next[s])

        # Cycle-centric layout also for PNG/SVG
        if CYCLE_CENTRIC_LAYOUT and cycle_list:
            posH = _layout_cycle_centric_tree(H, list(cycle_list)) if CYCLE_LAYOUT_MODE=="tree" \
                   else _layout_cycle_centric(H, list(cycle_list))
        else:
            posH = _layout_positions(H)

        fig = plt.figure(figsize=(15, 15)); ax = plt.gca()
        node_edgecolors = [CYCLE_BORDER_COLOR if v in cycle_set else NONCYCLE_BORDER for v in H.nodes()]
        node_lw         = [2 if v in cycle_set else 0.5 for v in H.nodes()]
        node_colors     = [ATTRACTOR_COLORS[id_map[aid]] for _ in H.nodes()]
        nx.draw_networkx_nodes(
            H, posH,
            node_color=node_colors,
            edgecolors=node_edgecolors, linewidths=node_lw,
            node_size=220, ax=ax
        )
        nx.draw_networkx_edges(H, posH, arrowstyle='-|>', arrowsize=10, width=1.2, alpha=0.85, ax=ax)

        if SHOW_NUM_LABELS:
            if LABEL_MODE == "bits":
                labels = {v: ''.join(str(b) for b in v) for v in H.nodes()}
            else:
                labels = {v: str(i) for i, v in enumerate(H.nodes())}
            txt = nx.draw_networkx_labels(H, posH, labels=labels, font_size=9, ax=ax)
            try:
                import matplotlib.patheffects as pe
                for t in txt.values():
                    t.set_path_effects([pe.withStroke(linewidth=2, foreground="white")])
            except Exception:
                pass

        ax.set_aspect('equal'); ax.axis('off')
        plt.tight_layout()

        def _slugify(s):
            return re.sub(r'[^A-Za-z0-9_\-]+', '_', s)

        slug = _slugify("plain")
        svg = f"{fname_prefix}_{slug}.svg"
        png = f"{fname_prefix}_{slug}.png"
        plt.savefig(svg, format='svg'); plt.savefig(png, dpi=300)
        plt.close(fig)
        created_files.extend([svg, png])

        # Per-basin HTML (cycle-centered)
        inter_html = f"{fname_prefix}_interactive_{slug}.html"
        write_interactive_html(
            H,
            inter_html,
            title=f"Basin of attractor {id_map[aid]} (no metrics)",
            cycle_nodes=cycle_set,
            cycle_order=list(cycle_list)
        )
        return svg, png, inter_html

    for new_idx, (aid, info) in enumerate(sorted_items, start=1):
        pref = os.path.join(separate_dir, f"{base_rules}_basin_{new_idx:02d}")
        svg, png, html = draw_basin(aid, info, pref)
        print(f"  Basin {new_idx} saved:")
        print(f"   • {svg}")
        print(f"   • {png}")
        print(f"   • {html}")

# ───────────────────────────────────────────────────────
# 8) Pack everything into a ZIP (preserving Energy_and_Frustration/ in paths)
# ───────────────────────────────────────────────────────
# Keep only existing files and deduplicate, ensure they live under OUTPUT_DIR
created_files = [
    p for p in dict.fromkeys(created_files)
    if os.path.isfile(p) and os.path.commonpath([os.path.abspath(p), os.path.abspath(OUTPUT_DIR)]) == os.path.abspath(OUTPUT_DIR)
]

zip_name = f"{base_rules}_Basin_Attraction.zip"
with zipfile.ZipFile(zip_name, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in created_files:
        # arcname keeps the Energy_and_Frustration/ prefix inside the zip
        arcname = os.path.relpath(p, start='.')  # retain OUTPUT_DIR/...
        zf.write(p, arcname=arcname)

print("\n  ZIP created with all outputs:")
print(f"   • {zip_name}")

if fallback_count:
    print(f"\n  {fallback_count} rule(s) used the 'fallback' compiler due to atypical characters/spaces.")
